# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

---

**1) What does one row mean for your lane?**

> One row = one **content item per day** at the raw warehouse grain (`report_date` × `client_hash_id` × `content_hash_id`). For modeling, we aggregate to one row per content item over a trailing feature window.

**2) Which table(s) will you use?**

> - `fact_content_daily_performance` — primary daily search + analytics metrics

> - `dim_content` — content metadata (age, word count, content type)

> - `dim_clients` — for client-holdout validation and history checks

**3) Which time window?**

> - **Feature window:** Prior 90 days of daily performance (days 1–90 before decision point)

> - **Label window:** Next 30 days after decision point (days 91–120) to observe true decline

> - **Development slice:** `month=2026-03` (mid-panel month — never the final month)

**4) What would you predict or rank (label or proxy)?**

> **Proxy label for this contract:** `has_search_clicks` = 1 if the page got any GSC clicks in the month.

> **Real target (capstone):** `is_declining` = 1 if impressions drop >20% in the future 30-day window vs. prior 30 days.

**5) What do you deliberately exclude, and why?**

> - `gsc_clicks` as a feature when the label is click-derived — it is the exact source of the label = **leakage**

> - `trend_direction` / `trend_pct` — computed from the label window itself, never features

> - Any metric from the future label window (days 91–120) — information from the future cannot be a feature

> - Product decision flags (`health_score`, `priority_score`) — not in data by design, but if rebuilt, never use as discovery features

> - Raw identifiers (`content_hash_id`, `client_hash_id`) as model features — for grouping/splitting only

In [1]:
# Setup: DuckDB + Hugging Face warehouse access
import duckdb
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import getpass
import os

print("Libraries imported. Ready to connect.")

Libraries imported. Ready to connect.


In [5]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [10]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

# Fix 1: Named secret
conn.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# Test 1: Can we read ANYTHING from HF?
print("Test 1: Reading a public dataset...")
try:
    pub = conn.execute("SELECT COUNT(*) FROM 'hf://datasets/datasets-examples/doc-formats-parquet-1/data/train-00000-of-00001.parquet'").df()
    print(f"  Public dataset OK: {pub.iloc[0,0]} rows")
except Exception as e:
    print(f"  Public dataset FAILED: {e}")

# Test 2: Can we list files in FlyRank dataset?
print("\nTest 2: Listing FlyRank dataset files...")
try:
    files = conn.execute("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/*')").df()
    print(f"  Found {len(files)} top-level items")
    print(files.head(20).to_string())
except Exception as e:
    print(f"  Listing FAILED: {e}")

# Test 3: Try direct HTTPS with Bearer token
print("\nTest 3: Direct HTTPS access...")
try:
    conn.execute(f"SET httpfs_http_header='Authorization: Bearer {HF_TOKEN}';")
    direct = conn.execute("""
        SELECT COUNT(*) FROM read_parquet(
            'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/part-00000.parquet'
        )
    """).df()
    print(f"  Direct HTTPS OK: {direct.iloc[0,0]} rows")
except Exception as e:
    print(f"  Direct HTTPS FAILED: {e}")

Test 1: Reading a public dataset...
  Public dataset FAILED: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/datasets-examples/doc-formats-parquet-1/resolve/main/data/train-00000-of-00001.parquet' (HTTP 400)

Test 2: Listing FlyRank dataset files...
  Listing FAILED: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main' (HTTP 400)

Test 3: Direct HTTPS access...
  Direct HTTPS FAILED: Catalog Error: unrecognized configuration parameter "httpfs_http_header"

Did you mean: "http_proxy"


In [11]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

# Set the token as a header for HTTP requests
conn.execute(f"SET httpfs_http_header='Authorization: Bearer {HF_TOKEN}';")

# Use direct HTTPS URLs instead of hf://
# The dataset files are accessible via:
# https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/...

# First, let's try to read a single file to verify access
test = conn.execute("""
    SELECT COUNT(*) as cnt
    FROM read_parquet('https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/part-00000.parquet')
""").df()
print(f"Test read: {test['cnt'].iloc[0]} rows")

CatalogException: Catalog Error: unrecognized configuration parameter "httpfs_http_header"

Did you mean: "http_proxy"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

---

| Field | Bucket | Why |
|---|---|---|
| `report_date` | **Context** | For window alignment and time-aware splits — never a model feature. |
| `client_hash_id` | **Context** | For client-holdout validation and grouping — never a model feature. |
| `content_hash_id` | **Context** | For joining and deduplication — never a model feature. |
| `gsc_impressions` | **Feature** | Search impressions in the feature window — knowable before prediction. |
| `gsc_avg_position` | **Feature** | Average search position in the feature window — knowable before prediction. |
| `ga4_sessions` | **Feature** | Site sessions in the feature window — knowable before prediction. |
| `ga4_scroll_events` | **Feature** | Engagement signal in the feature window — knowable before prediction. |
| `ga4_ai_sessions` | **Feature** | AI-referred sessions in the feature window — knowable before prediction. |
| `ga4_data_available` | **Context** | Flag for filtering — rows before GA4 start have zero-filled metrics. |
| `gsc_clicks` | **Excluded** | Source of the proxy label `has_search_clicks`. Using it as a feature = **leakage**. |
| `content_age_days` (dim_content) | **Feature** | Content age at decision moment — knowable before prediction. |
| `is_declining` (future window) | **Label** | True target: observed future decline over next 30 days. |
| `has_search_clicks` (proxy) | **Label** | Proxy label for this contract: any clicks in the month. |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# Query 1: Grain probe — verify one row = one (report_date, client, content)
grain_check = conn.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) as c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print("Grain check result:")
print(grain_check)

if len(grain_check) == 0:
    print("\n✅ GRAIN HOLDS: No duplicates found. One row = one content item per day.")
else:
    print(f"\n❌ GRAIN BROKEN: {len(grain_check)} duplicate groups found.")

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

In [ ]:
# Query 2: Row count, distinct counts, and date span
count_span = conn.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_hash_id) as num_clients,
        COUNT(DISTINCT content_hash_id) as num_content_items,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date,
        COUNT(DISTINCT report_date) as num_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("Row count and date span:")
print(count_span.to_string(index=False))

total = count_span['total_rows'].iloc[0]
clients = count_span['num_clients'].iloc[0]
days = count_span['num_days'].iloc[0]
print(f"\n📊 This slice has {total:,} rows across {days} days from {clients} clients.")

In [ ]:
# Query 3: Availability — how many rows have real GA4 and GSC data?
availability = conn.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) as gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE AND gsc_impressions IS NOT NULL) as both_available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("Availability check (using IS TRUE):")
print(availability.to_string(index=False))

total = availability['total_rows'].iloc[0]
ga4 = availability['ga4_available_rows'].iloc[0]
gsc = availability['gsc_available_rows'].iloc[0]
both = availability['both_available_rows'].iloc[0]

print(f"\n🔍 Out of {total:,} total rows:")
print(f"   - {ga4:,} ({ga4/total*100:.1f}%) have GA4 data (ga4_data_available IS TRUE)")
print(f"   - {gsc:,} ({gsc/total*100:.1f}%) have GSC impression data")
print(f"   - {both:,} ({both/total*100:.1f}%) have BOTH GA4 and GSC data")
print(f"\n⚠️  {(total-ga4)/total*100:.1f}% of rows are GA4-zero-filled. Always filter with IS TRUE.")

In [ ]:
# Build the 5-feature frame from month=2026-03
# Aggregate daily grain to one row per content item for the month.

feature_query = """
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        -- Feature 1: total search impressions in March (feature window)
        SUM(f.gsc_impressions) as sum_gsc_impressions_mar,
        -- Feature 2: average position in March (excluding 0 = 'no data')
        AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0) as avg_gsc_position_mar,
        -- Feature 3: total GA4 sessions in March (only where GA4 is available)
        SUM(f.ga4_sessions) FILTER (WHERE f.ga4_data_available IS TRUE) as sum_ga4_sessions_mar,
        -- Feature 4: total scroll events in March
        SUM(f.ga4_scroll_events) FILTER (WHERE f.ga4_data_available IS TRUE) as sum_ga4_scroll_events_mar,
        -- Feature 5: max daily AI sessions in March
        MAX(f.ga4_ai_sessions) FILTER (WHERE f.ga4_data_available IS TRUE) as max_ga4_ai_sessions_mar,
        -- Label source (EXCLUDED from honest features — this is the leakage source)
        SUM(f.gsc_clicks) as sum_gsc_clicks_mar,
        -- Proxy label: did this page get ANY search clicks in March?
        (SUM(f.gsc_clicks) > 0)::INTEGER as has_search_clicks_label
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
    WHERE f.gsc_impressions IS NOT NULL
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING SUM(f.gsc_impressions) > 0
"""

features_df = conn.execute(feature_query).df()

print(f"Feature frame shape: {features_df.shape}")
print(f"Label distribution:\n{features_df['has_search_clicks_label'].value_counts()}")
print("\nFirst 5 rows:")
print(features_df.head())

In [ ]:
# Honest model: 5 features, no leakage
honest_feature_cols = [
    'sum_gsc_impressions_mar',
    'avg_gsc_position_mar',
    'sum_ga4_sessions_mar',
    'sum_ga4_scroll_events_mar',
    'max_ga4_ai_sessions_mar'
]

X = features_df[honest_feature_cols].fillna(0)
y = features_df['has_search_clicks_label']

# Client-aware split: hold out entire clients
clients = features_df['client_hash_id'].unique()
np.random.seed(42)
np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = features_df['client_hash_id'].isin(test_clients)

X_train = X[~test_mask]
X_test = X[test_mask]
y_train = y[~test_mask]
y_test = y[test_mask]

print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Train clients: {features_df.loc[~test_mask, 'client_hash_id'].nunique()} | Test clients: {features_df.loc[test_mask, 'client_hash_id'].nunique()}")
print(f"Train label rate: {y_train.mean():.3f} | Test label rate: {y_test.mean():.3f}")

model_honest = LogisticRegression(max_iter=1000, class_weight='balanced')
model_honest.fit(X_train, y_train)
y_prob_honest = model_honest.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, y_prob_honest)

print(f"\n🎯 HONEST AUC (5 features, no leakage): {honest_auc:.4f}")

In [ ]:
# Leakage experiment: add ONE label-derived feature
# The leaked feature: sum_gsc_clicks_mar — the exact mathematical source of the proxy label

X_leak = X.copy()
X_leak['sum_gsc_clicks_mar'] = features_df['sum_gsc_clicks_mar'].fillna(0)

X_leak_train = X_leak[~test_mask]
X_leak_test = X_leak[test_mask]

model_leak = LogisticRegression(max_iter=1000, class_weight='balanced')
model_leak.fit(X_leak_train, y_train)
y_prob_leak = model_leak.predict_proba(X_leak_test)[:, 1]
leak_auc = roc_auc_score(y_test, y_prob_leak)

print(f"🚨 LEAKED AUC (5 features + sum_gsc_clicks_mar): {leak_auc:.4f}")
print(f"\n📈 AUC jump: {leak_auc - honest_auc:.4f} ({(leak_auc - honest_auc) / honest_auc * 100:.1f}% relative increase)")
print("\nThis jump is NOT model skill. It is the model reading the answer key.")
print("The 'leaked' feature is derived from the same calculation as the label.")

In [ ]:
# Delete the leak — keep only honest features and the honest number
print("=== FINAL HONEST RESULT ===")
print(f"Features used: {honest_feature_cols}")
print(f"Excluded (leakage): sum_gsc_clicks_mar — derived from the label source")
print(f"Label: has_search_clicks_label (proxy for 'page got search clicks')")
print(f"Split: Client-holdout ({len(test_clients)} clients held out)")
print(f"\n✅ HONEST AUC: {honest_auc:.4f}")
print(f"❌ LEAKED AUC (for demonstration only): {leak_auc:.4f}")
print(f"\nThe honest score is the only one we trust. The leaked score is a trap.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---

**Named limitation: Unbalanced panel + proxy label weakness**

1. **Unbalanced panel:** Different clients have different history depths. Some clients have 17 months of data; others have only 3. A global 90-day feature window works poorly for clients with shallow history. The fix is per-client windows anchored to `gsc_data_start` from `dim_clients`, but that adds complexity.

2. **GSC-only early rows:** Before a client's `ga4_data_start`, GA4 columns are zero-filled with `ga4_data_available = FALSE`. In our March 2026 slice this is less severe, but for historical windows it matters. We must always filter with `IS TRUE` and never treat zeros as 'no engagement'.

3. **Proxy label weakness:** Our label `has_search_clicks` is a simple binary proxy. It is NOT the ideal capstone label. A stronger label would predict future decline over a 30-day window, but that requires multi-month aggregation and strict window alignment — beyond this contract exercise.

4. **Single-month development:** We built features from March 2026 only. A real model would aggregate 90 days of daily data. This slice is for query mechanics and contract verification only.

5. **Sparse AI sessions:** `ga4_ai_sessions` is non-zero for only ~0.04% of rows in the full warehouse. Feature 5 is mostly zero and adds little signal. In a real model, we might drop it or use it as a binary flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.